# Data download and steps before using this notebook


- create data folder

- download all the data into the data folder

- see notes_about_data.txt to find out data that was used in this analysis

- TCIA files require NBIA data retriever and manually starting the data retrieval by double clicking on TCIA file (were not used in this study)

In [ ]:
# import libraries
import os, glob
import sys
import zipfile
import tarfile
import re
import nibabel as nib
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import nibabel as nib
import SimpleITK as sitk

In [ ]:
# directory to extract zip files
data_dir = 'data'
extract_dir = 'data'

# Compile the dataframe with filepaths

In [ ]:
# Root directory
root_dir = r"data\kits19"

dataset_list = []

# Find all case folders
case_dirs = glob.glob(os.path.join(root_dir, "case_*"))

for case_path in case_dirs:
    case_name = os.path.basename(case_path)           # case_00000
    case_number = case_name.split('_')[-1]                # 00000
    case_value= int(case_number)
    dataset_name = os.path.basename(os.path.dirname(case_path))  # kits19
    
    input_path = os.path.join(case_path, "imaging.nii.gz")
    label_path = os.path.join(case_path, "segmentation.nii.gz")
    
    # Check if both files exist
    if os.path.exists(input_path) and os.path.exists(label_path):
        dataset_list.append({
            "dataset": dataset_name,
            "case": case_value,
            "object": "kidney",
            "input": input_path,
            "label": label_path
        })

# Print result
kits_files = dataset_list
# for item in dataset_list:
#     print(item)

In [ ]:
import os
import glob

root_dir = r"data\sliver"

input_dir = os.path.join(root_dir, "input")
label_dir = os.path.join(root_dir, "label")

dataset_list = []

# Get all input .mhd files
input_files = sorted(glob.glob(os.path.join(input_dir, "liver-orig*.mhd")))

for input_path in input_files:
    
    # Extract case number (e.g., 001)
    filename = os.path.basename(input_path)
    case_number = filename.replace("liver-orig", "").replace(".mhd", "")
    case_value = int(case_number)  # Convert to integer for sorting if needed
    
    # Build corresponding label path
    label_path = os.path.join(label_dir, f"liver-seg{case_number}.mhd")
    
    if os.path.exists(label_path):
        dataset_list.append({
            "dataset": "sliver",
            "case": case_value,
            "object": "liver",
            "input": input_path,
            "label": label_path
        })

# Print result
sliver_files = dataset_list
# for item in dataset_list:
#     print(item)

In [ ]:
import re

# Root folder containing all tasks
root_base = r"data"

# List of all tasks you mentioned
task_folders = [
    "Task01_BrainTumour",
    "Task02_Heart",
    "Task03_Liver",
    "Task04_Hippocampus",
    "Task05_Prostate",
    "Task06_Lung",
    "Task07_Pancreas",
    "Task08_HepaticVessel",
    "Task09_Spleen",
    "Task10_Colon"
]

dataset_list = []

for task in task_folders:
    task_path = os.path.join(root_base, task)
    dataset_name = task
    object_name = task.split("_", 1)[1]  # Extract after first underscore
    
    input_dir = os.path.join(task_path, "input")
    label_dir = os.path.join(task_path, "label")
    
    # Get all input files (assuming .nii.gz)
    input_files = glob.glob(os.path.join(input_dir, "*.nii.gz"))
    
    for input_path in input_files:
        filename = os.path.basename(input_path)
        
        # Extract case number using digits in filename
        match = re.search(r'(\d+)', filename)
        case_number = int(match.group(1)) if match else None
        
        # Build corresponding label path
        label_filename = filename.replace("_", "").replace("._", "").replace("orig", "").replace(".nii.gz", "")
        label_path_pattern = os.path.join(label_dir, f"*{case_number}*.nii.gz")
        label_files = glob.glob(label_path_pattern)
        
        if label_files:
            label_path = label_files[0]
            dataset_list.append({
                "dataset": dataset_name,
                "object": object_name,
                "case": case_number,
                "input": input_path,
                "label": label_path
            })

# Print result
tasks_files = dataset_list
# for item in dataset_list:
#     print(item)

In [ ]:
all_files = sliver_files + kits_files + tasks_files
df = pd.DataFrame(all_files)

df.to_excel("medical_segmentation_datasets_filepaths.xlsx", index=False)

# Display some inputs

In [ ]:
df = pd.read_excel("medical_segmentation_datasets_filepaths.xlsx")

In [ ]:
def load_image(path):
    # if path.endswith(".nii") or path.endswith(".nii.gz"):
    #     img = nib.load(path)
    #     return img.get_fdata()
    # elif path.endswith(".mhd"):
    img = sitk.ReadImage(path)
    return sitk.GetArrayFromImage(img)  # shape: [z,y,x]
    # else:
    #     raise ValueError(f"Unsupported file: {path}")

def show_image_and_label(img, label, slice_idx=None):
    slice_idx_text = slice_idx
    if slice_idx is None:
        slice_idx = img.shape[0] // 2  # middle slice
        slice_idx_text = "middle"

    plt.figure(figsize=(12,5))
    plt.subplot(1,2,1)
    plt.imshow(img[slice_idx], cmap='gray')
    plt.title(f"Image (slice {slice_idx_text})")
    # plt.axis("off")

    plt.subplot(1,2,2)
    plt.imshow(label[slice_idx], cmap='jet', alpha=0.5)
    plt.title("Label overlay")
    # plt.axis("off")
    plt.show()

In [ ]:
unique_datasets = df['dataset'].unique()

for unique_dataset in unique_datasets:
    print(f"\nDataset: {unique_dataset}")
    df_uniq_dataset = df[df['dataset'] == unique_dataset]

    input_path = df_uniq_dataset.iloc[0]['input']
    label_path = df_uniq_dataset.iloc[0]['label']

    img_volume = sitk.ReadImage(input_path)

    input_spacing = img_volume.GetSpacing()
    input_size = img_volume.GetSize()

    img = load_image(input_path)
    label = load_image(label_path)
    print(f"Image shape: {img.shape}, Label shape: {label.shape}, unique labels: {np.unique(label)}")
    print(f"Original spacing: {input_spacing}, Original size: {input_size}")


    show_image_and_label(img, label)

    # use torchvision to resize image and label to 256x256; make sure the input and label shape is (D,H,W)
    # source: https://discuss.pytorch.org/t/3d-ct-images-resizing-and-resampling/104115/7

In [ ]:
df['dataset'].value_counts()